In [219]:
import pandas as pd

In [220]:
def read_npi():
    df = pd.read_csv("../data/input/california_npi_output.csv")
    return df

def read_lear():
    df = pd.read_csv("../data/input/lear_input.csv", sep="\t")
    return df 

def filter_state(df):
    df = df[df.STATE.str.contains(r"^CA$")]
    return df 

def fix_col_name(df):
    df = df.rename(columns={"NAME": "agency_name"})
    return df 

def normalize_npi(df):
    df.loc[:, "agency_name"] = (df.agency_name
                                .str.lower()
                                .str.strip()
                                .str.replace(r" pd$", " police department", regex=True)
                                .str.replace(r"co s[do]", "county sheriff's office", regex=True)
                                .str.replace(r" dps", " department of public safety", regex=True)
                                .str.replace(r"^ca ", "california ", regex=True)
                                .str.replace(r" co\b", " county", regex=True)
                                .str.replace(r"county(.+)coroner", r"county sheriff's office", regex=True)
                                .str.replace(r"county so$", "county sheriff's office", regex=True)
    )
  
    return df


def normalize_lear(df):
    df.loc[:, "agency_name"] = (df.agency_name
                                .str.lower()
                                .str.strip()
                                .str.replace(r"county(.+)coroner", r"county sheriff's office", regex=True)
    )
    return df 

dfa = read_npi()
dfb = read_lear()

/var/folders/r9/3_1rmy995xs_9z4vz66rsf9r0000gn/T/ipykernel_14088/3247836697.py:2: DtypeWarning: Columns (0,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/input/california_npi_output.csv")


In [221]:
dfb = dfb.pipe(filter_state).pipe(fix_col_name).pipe(normalize_lear)

dfb

,LEAR_ID,agency_name,STREET_ADDRESS,CITY,ZIP,STATE,COUNTY,FIPS,SOURCE,CSLLEA08_ID,...,PE14_MSA,PE14_POPULATION,PE14_MALE_OFFICERS,PE14_MALE_CIVILIANS,PE14_MALE_TOTAL,PE14_FEMALE_OFFICERS,PE14_FEMALE_CIVILIANS,PE14_FEMALE_TOTAL,PE14_TOTAL_EMPLOYEES,NO_POLICING
201,635867,pleasanton police department,PO BOX 909,Pleasanton,94566,CA,-3,6001,"CSLLEA14,POST,CSLLEA08,LEAIC",13430640,...,783,75060,73,4,77,8,28,36,113,False
202,635959,san leandro police department,901 E 14TH ST,San Leandro,94577,CA,-3,6001,"CSLLEA14,POST,CSLLEA08,LEAIC",11489190,...,783,88690,87,9,96,6,34,40,136,False
203,635426,angels camp police department,PO BOX 459,Angels Camp,95222,CA,-3,6009,"CSLLEA14,POST,CSLLEA08,LEAIC",13083400,...,,3716,7,0,7,0,1,1,8,False
204,635633,fortuna police department,621 11TH ST,Fortuna,95540,CA,-3,6023,"CSLLEA14,POST,CSLLEA08,LEAIC",13297620,...,,11752,13,1,14,1,7,8,22,False
205,635729,lassen county sheriff's office,1415 SHERIFF CADY LN,Susanville,96130,CA,-3,6035,"CSLLEA14,POST,CSLLEA08,LEAIC",13984190,...,,16657,47,7,54,14,14,28,82,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15748,636130,woodland police department,1000 LINCOLN AVE,Woodland,95695,CA,YOLO,6113,"CSLLEA14,POST,CSLLEA08,LEAIC",13599450,...,740,56854,52,0,52,9,14,23,75,False
15749,636119,west sacramento police department,550 JEFFERSON AVE,West Sacramento,95605,CA,YOLO,6113,"CSLLEA14,POST,CSLLEA08,LEAIC",13754470,...,740,50152,58,3,61,7,18,25,86,False
15797,636139,yuba county sheriff's office,PO BOX 1389,Marysville,95901,CA,YUBA,6115,"CSLLEA14,POST,CSLLEA08,LEAIC",12809220,...,990,58227,100,9,109,25,34,59,168,False
15798,635770,marysville police department,PO BOX 670,Marysville,95901,CA,YUBA,6115,"CSLLEA14,POST,CSLLEA08,LEAIC",13697060,...,990,12242,16,4,20,0,5,5,25,False


In [222]:
dfb_copy = dfb.copy()

dfb_copy = dfb_copy[dfb_copy.agency_name.str.contains(r"los angeles")]

dfb_copy 

,LEAR_ID,agency_name,STREET_ADDRESS,CITY,ZIP,STATE,COUNTY,FIPS,SOURCE,CSLLEA08_ID,...,PE14_MSA,PE14_POPULATION,PE14_MALE_OFFICERS,PE14_MALE_CIVILIANS,PE14_MALE_TOTAL,PE14_FEMALE_OFFICERS,PE14_FEMALE_CIVILIANS,PE14_FEMALE_TOTAL,PE14_TOTAL_EMPLOYEES,NO_POLICING
207,635751,los angeles police department,100 W 1ST ST,Los Angeles,90012,CA,-3,6037,"CSLLEA14,POST,CSLLEA08,LEAIC",13545150,...,480,3906772,8026,1115,9141,1881,1752,3633,12774,False
9017,635749,los angeles county sheriff's department,"4700 W. RAMONA BLVD., UNIT 1 4700 W. RAMONA BLVD.",MONTEREY PARK,91754,CA,LOS ANGELES,6037,"CSLLEA14,POST,CSLLEA08,LEAIC",13080500,...,480,1111939,7561,2936,10497,1659,4603,6262,16759,False


In [223]:
dfa = dfa.pipe(normalize_npi)

df = pd.merge(dfa, dfb, on="agency_name")

df

,person_nbr,first_name,middle_name,last_name,agency_name,start_date,end_date,separation_reason,LEAR_ID,STREET_ADDRESS,...,PE14_MSA,PE14_POPULATION,PE14_MALE_OFFICERS,PE14_MALE_CIVILIANS,PE14_MALE_TOTAL,PE14_FEMALE_OFFICERS,PE14_FEMALE_CIVILIANS,PE14_FEMALE_TOTAL,PE14_TOTAL_EMPLOYEES,NO_POLICING
0,A52-V94,CARLOS,ALBERTO,IRIARTE,san diego police department,1984-09-21,1984-10-05,Resigned,635934,1401 BROADWAY,...,777,1368690,1586,228,1814,290,416,706,2520,False
1,A52-V94,CARLOS,ALBERTO,IRIARTE,la palma police department,1994-09-22,1995-09-27,Resigned,635726,7792 WALKER ST,...,478,15977,21,0,21,1,8,9,30,False
2,A52-V94,CARLOS,ALBERTO,IRIARTE,redondo beach police department,1995-10-31,2009-10-04,Resigned,635883,PO BOX 639,...,480,68075,82,14,96,8,39,47,143,False
3,A52-V94,CARLOS,ALBERTO,IRIARTE,el segundo police department,2009-10-05,2010-01-30,Resigned,635615,348 MAIN ST,...,480,16990,56,5,61,4,10,14,75,False
4,B44-V58,MICHAEL,A,PUGLIESE,san mateo county sheriff's office,1973-02-01,2010-03-30,Retired,635965,400 COUNTY CTR,...,784,138997,402,57,459,60,119,179,638,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336652,A07-Q39,JAMES,J,MRAZ,westminster police department,1983-02-04,2007-07-20,Retired,636117,8200 WESTMINSTER BLVD,...,478,92217,80,4,84,9,30,39,123,False
336653,B82-Q06,RACHAEL,LICHT,VANSLOTEN,oakland police department,1996-07-22,1997-01-31,Promotion/Demotion,635823,455 7TH ST,...,783,409994,632,115,747,83,264,347,1094,False
336654,B82-Q06,RACHAEL,LICHT,VANSLOTEN,oakland police department,1997-01-31,2021-07-24,Retired,635823,455 7TH ST,...,783,409994,632,115,747,83,264,347,1094,False
336655,A91-J57,GEORGE,JOSEPH,MAHONEY,san mateo county sheriff's office,1983-10-25,2005-12-29,Resigned,635965,400 COUNTY CTR,...,784,138997,402,57,459,60,119,179,638,False
